In [5]:
pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
You should consider upgrading via the '/Users/nirashanelki/Documents/Works/Research/codes/factual_inconsistency_validate_xai/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [1]:
"""
Study 1 — Human Evaluation Analysis
KG-Based Explanation Plausibility Assessment

Items (8 total):
  Item 01: discovery → involves → skull              [Direct, short]
  Item 02: Abdullah Hayayei → was hit during → ...   [Direct, long, disconnected]
  Item 03: website → flooded with → auto comments    [Direct, half-complete chain]
  Item 04: legal challenge → against → student ...   [Direct, hard text]
  Item 05: study → suggests → displacement ...       [Direct, hard text]
  Item 06: closure plans → called in by → Scot. gov  [Direct, long but easy]
  Item 07: 1000+ homeless → referred to → charity    [Direct, long but easy]
  Item 08: Kazuyoshi Miura → signed → contract ext   [UNSUPPORTED]

Questions per item (7-point Likert, 1=Strongly Disagree, 7=Strongly Agree):
  Q1: Usefulness    — "This explanation helped me verify the facts..."
  Q2: Understand.   — "I can clearly see the logical path..."
  Q3: Trustworth.   — "I feel confident the system isn't making up..."
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
from scipy import stats

# ── Configuration ─────────────────────────────────────────────────────────────
FILE = "./Copy of Understanding AI Summary Explanations_ A Brief User Study (Responses).xlsx"
OUT  = "./outputs/"

ITEMS = {
    1: {"label": "Item 01\ndiscovery→skull",           "type": "Direct",      "difficulty": "Easy"},
    2: {"label": "Item 02\nHayayei→hit",               "type": "Direct",      "difficulty": "Hard"},
    3: {"label": "Item 03\nwebsite→flooded",           "type": "Direct",      "difficulty": "Medium"},
    4: {"label": "Item 04\nlegal challenge",           "type": "Direct",      "difficulty": "Hard"},
    5: {"label": "Item 05\nstudy→displacement",        "type": "Direct",      "difficulty": "Hard"},
    6: {"label": "Item 06\nclosure plans",             "type": "Direct",      "difficulty": "Easy"},
    7: {"label": "Item 07\nhomeless→charity",          "type": "Direct",      "difficulty": "Easy"},
    8: {"label": "Item 08\nMiura→contract",            "type": "Unsupported", "difficulty": "Easy"},
}

DIMS = {
    "usefulness":        "This explanation helped me verify the facts in the summary easily.",
    "understandability": "I can clearly see the logical path the model took to reach its conclusion.",
    "trustworthiness":   "I feel confident that the system isn't just 'making up' an explanation.",
}

COLOURS = {
    "usefulness":        "#3b82f6",
    "understandability": "#10b981",
    "trustworthiness":   "#f59e0b",
}

plt.rcParams.update({
    "font.family":    "serif",
    "font.size":      10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "figure.dpi":     150,
    "savefig.dpi":    300,
    "savefig.bbox":   "tight",
})

# ── Load and reshape data ──────────────────────────────────────────────────────
df_raw = pd.read_excel(FILE)
n_participants = len(df_raw)
print(f"Participants: {n_participants}")

# Build tidy dataframe: one row per (participant × item × dimension)
records = []
for item_num in range(1, 9):
    suffix = "" if item_num == 1 else f" {item_num}"
    for dim_key, dim_col_base in DIMS.items():
        col = dim_col_base + suffix
        if col in df_raw.columns:
            for p_idx, val in enumerate(df_raw[col]):
                records.append({
                    "participant": p_idx,
                    "item":        item_num,
                    "dimension":   dim_key,
                    "score":       val,
                    "type":        ITEMS[item_num]["type"],
                    "difficulty":  ITEMS[item_num]["difficulty"],
                })

df = pd.DataFrame(records)
df["score"] = pd.to_numeric(df["score"], errors="coerce")
print(f"\nTotal data points: {len(df)} (expected {n_participants * 8 * 3} = {n_participants*24})")
print(f"Missing values: {df['score'].isna().sum()}")

# ── Summary statistics ─────────────────────────────────────────────────────────
print("\n" + "="*70)
print("OVERALL DIMENSION STATISTICS")
print("="*70)
overall = df.groupby("dimension")["score"].agg(["mean", "std", "median", "min", "max"])
overall.columns = ["Mean", "SD", "Median", "Min", "Max"]
print(overall.round(3))

print("\n" + "="*70)
print("PER-ITEM STATISTICS (mean across all 3 dimensions)")
print("="*70)
per_item = df.groupby(["item"])["score"].agg(["mean", "std"]).round(3)
per_item.columns = ["Mean", "SD"]
for i, row in per_item.iterrows():
    print(f"  Item {i:02d} [{ITEMS[i]['type']:11s}]: Mean={row['Mean']:.3f}, SD={row['SD']:.3f}")

print("\n" + "="*70)
print("PER-ITEM × PER-DIMENSION STATISTICS")
print("="*70)
per_item_dim = df.groupby(["item", "dimension"])["score"].agg(["mean", "std"]).round(3)
print(per_item_dim.to_string())

print("\n" + "="*70)
print("BY SUPPORT TYPE (Supported vs Unsupported)")
print("="*70)
by_type = df.groupby(["type", "dimension"])["score"].agg(["mean", "std"]).round(3)
print(by_type.to_string())

# ── Statistical tests ──────────────────────────────────────────────────────────
print("\n" + "="*70)
print("STATISTICAL TESTS")
print("="*70)

# Overall reliability: Cronbach's alpha approximation
print("\nCronbach's Alpha per dimension (inter-item reliability):")
for dim in DIMS.keys():
    scores_matrix = []
    for item_num in range(1, 9):
        item_scores = df[(df["item"] == item_num) & (df["dimension"] == dim)]["score"].values
        if len(item_scores) == n_participants:
            scores_matrix.append(item_scores)
    if scores_matrix:
        k = len(scores_matrix)
        item_vars = [np.var(s, ddof=1) for s in scores_matrix]
        total_scores = np.sum(scores_matrix, axis=0)
        total_var = np.var(total_scores, ddof=1)
        alpha = (k / (k - 1)) * (1 - sum(item_vars) / total_var)
        print(f"  {dim:20s}: α = {alpha:.3f}")

# Wilcoxon: supported items (1-7) vs unsupported item (8)
print("\nWilcoxon signed-rank: Supported items vs Unsupported (Item 08):")
for dim in DIMS.keys():
    supported_scores = df[(df["item"] != 8) & (df["dimension"] == dim)].groupby("participant")["score"].mean()
    unsupported_scores = df[(df["item"] == 8) & (df["dimension"] == dim)]["score"]
    
    # Align by participant
    supported_arr = supported_scores.values
    unsupported_arr = unsupported_scores.values
    
    if len(supported_arr) == len(unsupported_arr):
        stat, p = stats.wilcoxon(supported_arr, unsupported_arr)
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
        print(f"  {dim:20s}: W={stat:.1f}, p={p:.4f} {sig}")

# Kruskal-Wallis across items per dimension
# print("\nKruskal-Wallis across 8 items per dimension:")
# for dim in DIMS.keys():
#     groups = [df[(df["item"] == i) & (df["dimension"] == dim)]["score"].values 
#               for i in range(1, 9)]
#     stat, p = stats.kruskal(*groups)
#     sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
#     print(f"  {dim:20s}: H={stat:.3f}, p={p:.4f} {sig}")

# Friedman test across items (repeated measures)
print("\nFriedman test across 8 items per dimension (repeated measures):")

for dim in DIMS.keys():
    # Build participant × item matrix
    pivot = df[df["dimension"] == dim].pivot(
        index="participant",
        columns="item",
        values="score"
    )
    
    # Ensure complete cases (required for Friedman)
    pivot = pivot.dropna()
    
    if pivot.shape[1] == 8:
        # Each column = item, each row = participant
        stat, p = stats.friedmanchisquare(*[pivot[i] for i in range(1, 9)])
        
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
        
        print(f"  {dim:20s}: χ²={stat:.3f}, p={p:.4f} {sig}")
    else:
        print(f"  {dim:20s}: Skipped (missing data)")

# ── Figure 1: Mean scores per dimension overall ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(12, 4.5), sharey=True)

for ax, (dim, colour) in zip(axes, COLOURS.items()):
    item_means = []
    item_sds   = []
    item_labels = []
    
    for item_num in range(1, 9):
        scores = df[(df["item"] == item_num) & (df["dimension"] == dim)]["score"].dropna()
        item_means.append(scores.mean())
        item_sds.append(scores.std())
        item_labels.append(f"I{item_num}")
    
    x = np.arange(len(item_means))
    
    # Colour item 8 differently (unsupported)
    bar_colours = [colour] * 7 + ["#ef4444"]
    
    bars = ax.bar(x, item_means, color=bar_colours, alpha=0.85,
                  edgecolor="white", linewidth=0.8, zorder=2)
    ax.errorbar(x, item_means, yerr=item_sds, fmt="none",
                color="#374151", capsize=3, linewidth=1.2, zorder=3)
    
    # Value labels
    for bar, mean in zip(bars, item_means):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.08,
                f"{mean:.1f}",
                ha="center", va="bottom", fontsize=7.5, color="#374151")
    
    ax.set_xticks(x)
    ax.set_xticklabels(item_labels, fontsize=8)
    ax.set_ylim(0, 8.2)
    ax.set_yticks([1, 2, 3, 4, 5, 6, 7])
    ax.axhline(4, color="#94a3b8", linewidth=0.8, linestyle="--", alpha=0.7)
    ax.set_title(dim.capitalize(), fontweight="bold", pad=8)
    ax.set_xlabel("Item", labelpad=5)
    ax.grid(axis="y", alpha=0.3, zorder=1)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

axes[0].set_ylabel("Mean Likert Score (1–7)", labelpad=8)

# Legend
supported_patch = mpatches.Patch(color="#3b82f6", alpha=0.85, label="Supported items (I1–I7)")
unsupported_patch = mpatches.Patch(color="#ef4444", alpha=0.85, label="Unsupported item (I8)")
fig.legend(handles=[supported_patch, unsupported_patch],
           loc="lower center", ncol=2, fontsize=9,
           bbox_to_anchor=(0.5, -0.05))

fig.suptitle("Study 1 — Mean Likert Scores per Item and Dimension\n"
             "(n=10 participants, error bars = ±1 SD)",
             fontweight="bold", y=1.02)

plt.tight_layout()
plt.savefig(OUT + "study1_per_item_scores.pdf")
plt.savefig(OUT + "study1_per_item_scores.png", dpi=200)
plt.close()
print("\nSaved: study1_per_item_scores.pdf")

# ── Figure 2: Overall dimension comparison (box + strip) ──────────────────────
fig, ax = plt.subplots(figsize=(7, 4.5))

dim_labels = list(DIMS.keys())
positions  = np.arange(len(dim_labels))

for i, (dim, colour) in enumerate(COLOURS.items()):
    scores = df[df["dimension"] == dim]["score"].dropna().values
    
    bp = ax.boxplot(scores, positions=[i], widths=0.4,
                    patch_artist=True, notch=False,
                    boxprops=dict(facecolor=colour, alpha=0.6, linewidth=1.2),
                    medianprops=dict(color="#1e293b", linewidth=2),
                    whiskerprops=dict(linewidth=1.2),
                    capprops=dict(linewidth=1.2),
                    flierprops=dict(marker="o", markersize=4, alpha=0.5))
    
    # Strip plot
    jitter = np.random.uniform(-0.12, 0.12, size=len(scores))
    ax.scatter(i + jitter, scores, color=colour, alpha=0.6, s=20, zorder=3)
    
    # Mean annotation
    ax.text(i, scores.mean() + 0.15, f"μ={scores.mean():.2f}",
            ha="center", fontsize=8.5, color="#1e293b", fontweight="bold")

ax.set_xticks(positions)
ax.set_xticklabels(["Usefulness", "Understandability", "Trustworthiness"], fontsize=10)
ax.set_ylim(0.5, 7.8)
ax.set_yticks([1, 2, 3, 4, 5, 6, 7])
ax.set_ylabel("Likert Score (1–7)", labelpad=8)
ax.axhline(4, color="#94a3b8", linewidth=0.8, linestyle="--", alpha=0.7, label="Neutral (4)")
ax.set_title("Study 1 — Overall Distribution by Evaluation Dimension\n"
             "(all items, n=10 participants)",
             fontweight="bold", pad=10)
ax.grid(axis="y", alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(fontsize=8.5)

plt.tight_layout()
plt.savefig(OUT + "study1_dimension_distribution.pdf")
plt.savefig(OUT + "study1_dimension_distribution.png", dpi=200)
plt.close()
print("Saved: study1_dimension_distribution.pdf")

# ── Figure 3: Heatmap — per item × per dimension means ────────────────────────
heatmap_data = df.groupby(["item", "dimension"])["score"].mean().unstack()
heatmap_data = heatmap_data[["usefulness", "understandability", "trustworthiness"]]

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(heatmap_data.values, cmap="RdYlGn", vmin=1, vmax=7, aspect="auto")

ax.set_xticks(range(3))
ax.set_xticklabels(["Usefulness", "Understandability", "Trustworthiness"], fontsize=10)
ax.set_yticks(range(8))
ax.set_yticklabels([
    f"I{i} [{ITEMS[i]['type'][:3]}]" for i in range(1, 9)
], fontsize=9)

# Annotate cells
for i in range(8):
    for j in range(3):
        val = heatmap_data.values[i, j]
        text_colour = "white" if val < 2.5 or val > 5.5 else "#1e293b"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                fontsize=9, color=text_colour, fontweight="bold")

cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Mean Likert Score", fontsize=9)
cbar.set_ticks([1, 2, 3, 4, 5, 6, 7])

ax.set_title("Study 1 — Per-Item × Per-Dimension Mean Scores\n"
             "(1 = Strongly Disagree, 7 = Strongly Agree)",
             fontweight="bold", pad=12)

# Highlight unsupported item row
for j in range(3):
    ax.add_patch(plt.Rectangle((j - 0.5, 6.5), 1, 1,
                                fill=False, edgecolor="#ef4444",
                                linewidth=2.5, zorder=4))

ax.text(3.15, 7, "← Unsupported", va="center", fontsize=8,
        color="#ef4444", fontweight="bold")

plt.tight_layout()
plt.savefig(OUT + "study1_heatmap.pdf")
plt.savefig(OUT + "study1_heatmap.png", dpi=200)
plt.close()
print("Saved: study1_heatmap.pdf")

# ── Print LaTeX table ──────────────────────────────────────────────────────────
print("\n" + "="*70)
print("LaTeX TABLE — Per-Item × Per-Dimension Means and SDs")
print("="*70)
print(r"\begin{table}[h]")
print(r"\centering")
print(r"\small")
print(r"\begin{tabular}{llccccc}")
print(r"\hline")
print(r"\textbf{Item} & \textbf{Type} & \multicolumn{2}{c}{\textbf{Usefulness}} & \multicolumn{2}{c}{\textbf{Understand.}} & \textbf{Trust.} \\")
print(r" & & M & SD & M & SD & M (SD) \\")
print(r"\hline")

for item_num in range(1, 9):
    item_data = df[df["item"] == item_num]
    means = item_data.groupby("dimension")["score"].mean()
    sds   = item_data.groupby("dimension")["score"].std()
    
    u_m  = means.get("usefulness", np.nan)
    u_sd = sds.get("usefulness", np.nan)
    un_m = means.get("understandability", np.nan)
    un_sd= sds.get("understandability", np.nan)
    t_m  = means.get("trustworthiness", np.nan)
    t_sd = sds.get("trustworthiness", np.nan)
    
    item_type = ITEMS[item_num]["type"]
    marker = r" \cellcolor{red!10}" if item_num == 8 else ""
    
    print(f"Item {item_num:02d}{marker} & {item_type} & "
          f"{u_m:.2f} & {u_sd:.2f} & "
          f"{un_m:.2f} & {un_sd:.2f} & "
          f"{t_m:.2f} ({t_sd:.2f}) \\\\")

print(r"\hline")

# Overall means
for dim_label, dim_key in [("Usefulness", "usefulness"), 
                            ("Understand.", "understandability"),
                            ("Trust.", "trustworthiness")]:
    all_scores = df[df["dimension"] == dim_key]["score"].dropna()
    print(f"% {dim_label}: Mean={all_scores.mean():.3f}, SD={all_scores.std():.3f}")

print(r"\end{tabular}")
print(r"\caption{Study 1 Likert results per item and dimension (n=10).}")
print(r"\label{tab:study1_results}")
print(r"\end{table}")

print("\n✓ Analysis complete. Figures saved to outputs/")

Participants: 10

Total data points: 240 (expected 240 = 240)
Missing values: 0

OVERALL DIMENSION STATISTICS
                    Mean     SD  Median  Min  Max
dimension                                        
trustworthiness    5.600  1.514     6.0    1    7
understandability  5.600  1.506     6.0    1    7
usefulness         5.812  1.388     6.0    1    7

PER-ITEM STATISTICS (mean across all 3 dimensions)
  Item 01 [Direct     ]: Mean=5.867, SD=1.042
  Item 02 [Direct     ]: Mean=5.900, SD=1.185
  Item 03 [Direct     ]: Mean=6.233, SD=1.194
  Item 04 [Direct     ]: Mean=6.033, SD=1.159
  Item 05 [Direct     ]: Mean=6.133, SD=0.681
  Item 06 [Direct     ]: Mean=5.500, SD=1.526
  Item 07 [Direct     ]: Mean=5.400, SD=1.380
  Item 08 [Unsupported]: Mean=4.300, SD=2.231

PER-ITEM × PER-DIMENSION STATISTICS
                        mean    std
item dimension                     
1    trustworthiness     5.8  1.229
     understandability   5.8  1.135
     usefulness          6.0  0.816
2  